# Tăng độ phủ vùng biển số với HOG + SVM
Phiên bản này kết hợp Sobel, Canny và phép black-hat hình thái với nhiều kernel. SVM học từ đầu bằng nhãn khung train, chấm vùng trước khi khử trùng lặp. Chạy trên CPU, không dùng pretrained.

Chạy từ gốc repo với `numpy`, `opencv-python`, `scikit-learn`. Bật `SMOKE_RUN` để kiểm tra nhanh. Kết quả ghi riêng trong `outputs/svm_optimized`.

In [ ]:
import csv
import json
import random
from pathlib import Path

import cv2
import numpy as np
from sklearn.svm import LinearSVC

from src.detector import (
    PlateDetector, iou, proposal_features, propose, propose_ensemble, suppress_overlaps,
)

DATASET_OVERRIDE = None
SMOKE_RUN = False
SEED = 42
IOU_THRESHOLD = 0.5
OUTPUT_DIR = Path('outputs/svm_optimized_smoke' if SMOKE_RUN else 'outputs/svm_optimized')

dataset_name = 'License Plate Detection Dataset'
roots = [
    Path(dataset_name),
    Path('../CuoiKy') / dataset_name,
]
kaggle_input = Path('/kaggle/input')
if kaggle_input.is_dir():
    roots.extend(sorted(kaggle_input.rglob(dataset_name)))
if DATASET_OVERRIDE is not None:
    roots = [Path(DATASET_OVERRIDE)]
root = next((p for p in roots if (p / 'images').is_dir() and (p / 'labels').is_dir()), None)
if root is None:
    raise FileNotFoundError('Dat DATASET_OVERRIDE den thu muc co images/ va labels/.')
print(root.resolve())

MAX_SIDE = 1024
MAX_BOXES = 5
NMS_IOU = 0.45
BASELINE_PROPOSAL = dict(method='sobel', kernel=[9, 3], min_area_ratio=0.002)
BRANCHES = [
    dict(method=method, kernel=kernel, min_area_ratio=0.0004, min_aspect=0.90,
         min_fill=0.20, min_density=0.05, overlap=0.8, nested=True)
    for method, kernels in [
        ('sobel', ([7, 3], [13, 5], [21, 9])),
        ('canny', ([7, 3], [13, 7])),
        ('blackhat', ([9, 3], [17, 7])),
    ]
    for kernel in kernels
]
ENSEMBLE = dict(branches=BRANCHES, per_branch=14, max_candidates=42)

def proposals(gray):
    return propose_ensemble(gray, ENSEMBLE, MAX_SIDE)

def baseline_proposals(gray):
    return propose(gray, BASELINE_PROPOSAL, max_side=768, max_boxes=5)

## 1. Nhãn và tập dữ liệu
Tập train dùng để học trọng số. Tập val chọn hệ số regularization `C` và ngưỡng SVM. Tập test dùng để so sánh cuối cùng. Danh sách loại trừ là các bản trùng pixel đã xác định trong EDA.

In [ ]:
EXACT_DUPLICATES = {
    'train': {'quandoi64.jpg', 'quandoi92.jpg', 'quandoi99.jpg'},
    'val': set(),
    'test': {'quandoi75.jpg'},
}
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

def image_paths(directory):
    return sorted(p for p in directory.iterdir() if p.suffix.lower() in IMAGE_SUFFIXES)

def split_paths(split):
    paths = image_paths(root / 'images' / split)
    paths = [p for p in paths if p.name not in EXACT_DUPLICATES[split]]
    if SMOKE_RUN:
        paths = random.Random(SEED).sample(paths, min(18, len(paths)))
    if not paths:
        raise ValueError(f'Tap {split} rong.')
    return paths

def read_gray(path):
    gray = cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise ValueError(f'Khong doc duoc: {path}')
    return gray

def ground_truth(path, split, image_shape):
    height, width = image_shape
    relative = path.relative_to(root / 'images' / split).with_suffix('.txt')
    label_path = root / 'labels' / split / relative
    boxes = []
    for row in label_path.read_text(encoding='utf-8-sig').splitlines():
        if not row.strip():
            continue
        label, x, y, w, h = map(float, row.split())
        if label != 0 or w <= 0 or h <= 0:
            raise ValueError(f'Nhan khong hop le: {label_path}')
        box = (
            max(0, (x - w / 2) * width),
            max(0, (y - h / 2) * height),
            min(width, (x + w / 2) * width),
            min(height, (y + h / 2) * height),
        )
        if box[0] >= box[2] or box[1] >= box[3]:
            raise ValueError(f'Khung nam ngoai anh: {label_path}')
        boxes.append(box)
    return boxes

train_paths = split_paths('train')
val_paths = split_paths('val')
test_paths = split_paths('test')
print(f'Train: {len(train_paths)} | Val: {len(val_paths)} | Test: {len(test_paths)}')

## 2. Huấn luyện trên vùng ứng viên mới
Ví dụ dương gồm vùng thật và ứng viên IoU ≥ 0,5. Giữ tối đa 10 ứng viên âm mỗi ảnh gồm các vùng xếp hạng cao và các vùng chọn ngẫu nhiên với seed cố định. Bỏ qua ứng viên có IoU nằm giữa 0,35 và 0,5.

In [ ]:
features, classes = [], []
rng = random.Random(SEED)
for path in train_paths:
    gray = read_gray(path)
    truth = ground_truth(path, 'train', gray.shape)
    for box in truth:
        features.append(proposal_features(gray, box))
        classes.append(1)
    negatives = []
    for box, _ in proposals(gray):
        overlap = max((iou(box, ref) for ref in truth), default=0)
        if overlap >= 0.5:
            features.append(proposal_features(gray, box))
            classes.append(1)
        elif overlap <= 0.35:
            negatives.append(box)
    tail = negatives[5:]
    for box in negatives[:5] + rng.sample(tail, min(5, len(tail))):
        features.append(proposal_features(gray, box))
        classes.append(0)

X_train = np.stack(features)
y_train = np.asarray(classes)
del features, classes
if set(y_train) != {0, 1}:
    raise ValueError('Can ca vi du bien so va vi du nhieu trong train.')
print(dict(negatives=int((y_train == 0).sum()), positives=int((y_train == 1).sum()),
           feature_count=X_train.shape[1]))

models = {}
for c in (0.01, 0.1, 1.0):
    model = LinearSVC(C=c, class_weight='balanced', dual=False, max_iter=5000, random_state=SEED)
    models[c] = model.fit(X_train, y_train)

## 3. Kiểm tra độ phủ và chọn ngưỡng trên val
Độ phủ là tỷ lệ biển có ít nhất một vùng ứng viên đạt IoU ≥ 0,5 trước SVM. Chọn `C` và ngưỡng dựa trên F1 sau SVM và NMS. Vùng điểm cao được giữ trước.

In [ ]:
def match_counts(predictions, truth):
    available = set(range(len(truth)))
    matched = 0
    for box, _ in predictions:
        if not available:
            break
        index = max(sorted(available), key=lambda i: iou(box, truth[i]))
        if iou(box, truth[index]) >= IOU_THRESHOLD:
            available.remove(index)
            matched += 1
    return np.array([matched, len(predictions) - matched, len(truth) - matched])

def metrics(counts):
    tp, fp, fn = map(int, counts)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0
    return dict(tp=tp, fp=fp, fn=fn, precision=precision, recall=recall, f1=f1)

def collect_split(paths, split):
    records = []
    feature_rows = []
    covered = {'sobel': 0, 'ensemble': 0}
    total = 0
    for path in paths:
        gray = read_gray(path)
        truth = ground_truth(path, split, gray.shape)
        candidates = proposals(gray)
        total += len(truth)
        for key, boxes in [('sobel', baseline_proposals(gray)), ('ensemble', candidates)]:
            covered[key] += sum(any(iou(ref, box) >= IOU_THRESHOLD for box, _ in boxes)
                                for ref in truth)
        start = len(feature_rows)
        feature_rows.extend(proposal_features(gray, box) for box, _ in candidates)
        records.append((truth, candidates, slice(start, len(feature_rows))))
    X = np.stack(feature_rows) if feature_rows else np.empty((0, X_train.shape[1]))
    return records, X, {key: count / total for key, count in covered.items()}

def evaluate(records, scores=None, threshold=None):
    counts = np.zeros(3, dtype=int)
    for truth, candidates, span in records:
        if scores is not None:
            ranked = [(box, float(score)) for (box, _), score in zip(candidates, scores[span])
                      if score >= threshold]
            candidates = suppress_overlaps(ranked, NMS_IOU, MAX_BOXES)
        counts += match_counts(candidates, truth)
    return metrics(counts)

val_records, X_val, val_coverage = collect_split(val_paths, 'val')
val_rows = []
for c, model in models.items():
    scores = model.decision_function(X_val)
    thresholds = sorted(set(np.r_[np.linspace(-0.5, 2.0, 26), np.quantile(scores, [0.65, 0.75, 0.85, 0.9])]))
    for threshold in thresholds:
        val_rows.append(dict(c=c, threshold=float(threshold),
                             **evaluate(val_records, scores, threshold)))
selected = max(val_rows, key=lambda row: (row['f1'], row['recall'], row['precision']))
print('Proposal coverage on val:', val_coverage)
print('SVM val:', selected)

## 4. Chấm detector đã chốt trên test
Chạy với cấu hình chọn từ val. Bảng so sánh với SVM cũ xuất thêm khi đã có artifact của notebook cơ sở. Độ phủ đo riêng trước bước SVM.

In [ ]:
model = models[selected['c']]
threshold = selected['threshold']
test_records, X_test, test_coverage = collect_split(test_paths, 'test')
test_scores = model.decision_function(X_test)
test_rows = [
    dict(method='ensemble_hog_svm', **evaluate(test_records, test_scores, threshold)),
]
old_config = Path('outputs/svm_detector/detector_config.json')
if old_config.is_file() and not SMOKE_RUN:
    baseline = PlateDetector(old_config)
    baseline_counts = np.zeros(3, dtype=int)
    for path in test_paths:
        gray = read_gray(path)
        baseline_counts += match_counts(baseline.predict(gray), ground_truth(path, 'test', gray.shape))
    test_rows.insert(0, dict(method='sobel_hog_svm', **metrics(baseline_counts)))
print('Proposal coverage on test:', test_coverage)
for row in test_rows:
    print(row)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for name, rows in [('validation.csv', val_rows), ('test.csv', test_rows)]:
    with (OUTPUT_DIR / name).open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

weights_name = 'svm_weights.npz'
np.savez(OUTPUT_DIR / weights_name, weights=model.coef_.ravel(), intercept=model.intercept_.item())
artifact = {
    'ensemble': ENSEMBLE,
    'nms_iou': NMS_IOU,
    'max_side': MAX_SIDE,
    'max_boxes': MAX_BOXES,
    'c': selected['c'],
    'threshold': threshold,
    'weights_file': weights_name,
    'eval_iou': IOU_THRESHOLD,
    'train_images': len(train_paths),
    'val_images': len(val_paths),
    'test_images': len(test_paths),
    'smoke_run': SMOKE_RUN,
    'val_proposal_coverage': val_coverage,
    'test_proposal_coverage': test_coverage,
}
artifact_path = OUTPUT_DIR / 'detector_config.json'
artifact_path.write_text(json.dumps(artifact, indent=2) + '\n', encoding='utf-8')

## 5. Kiểm tra nạp mô hình
Nạp lại file vừa xuất bằng lớp inference của dự án và so với kết quả SVM trên một số ảnh test.

In [ ]:
detector = PlateDetector(artifact_path)
indices = [i for i, (_, candidates, _) in enumerate(test_records) if candidates][:10]
assert indices, 'Khong co ung vien de kiem tra nap mo hinh.'
for index in indices:
    _, candidates, span = test_records[index]
    ranked = [(box, float(score)) for (box, _), score in zip(candidates, test_scores[span])
              if score >= threshold]
    expected = [box for box, _ in suppress_overlaps(ranked, NMS_IOU, MAX_BOXES)]
    predicted = detector.predict(read_gray(test_paths[index]))
    assert [box for box, _ in predicted] == expected
print(f'Checked saved model on {len(indices)} images.')
print(OUTPUT_DIR.resolve())